# Stage 5-3: Predictor, Visualizer, Logger 구현

이 노트북은 `src/core/predictor.py`의 `Predictor`, `src/core/visualizer.py`의 `Visualizer`,
`src/core/logger.py`의 `Logger`를 실습한다.

- `Predictor`: raw logit을 task별 예측값으로 후처리한다 (argmax / threshold / round_clip)
- `Visualizer`: 예측 결과와 입력 이미지를 grid로 저장한다
- `Logger`: epoch별 loss/metric 로그를 누적하고 CSV로 내보낸다

In [ ]:
# sys.path 설정 — 프로젝트 루트를 모듈 검색 경로에 추가한다.
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tempfile

In [ ]:
from src.data.mnist import MNISTDataset, get_task_spec
from src.data.dataloader import Dataloader
from src.models.mlp import MLP
from src.core.optimizers import SGD
from src.core.trainer import Trainer
from src.core.evaluator import Evaluator
from src.core.predictor import Predictor
from src.core.visualizer import Visualizer
from src.core.logger import Logger

## 1. Predictor — task별 예측 후처리

모델 `forward`의 출력은 activation이 없는 raw logit이다.
`Predictor`는 task별로 후처리하여 사람이 읽을 수 있는 예측값을 반환한다.

$$\text{logit} = f_\theta(x) \xrightarrow{\text{후처리}} \text{prediction}$$

| task | prediction_mode | 후처리 |
|---|---|---|
| `multiclass` | `argmax` | $\arg\max_c \hat{y}_c$ |
| `binary` | `threshold` | $\mathbf{1}[\sigma(\hat{y}) \geq 0.5]$ |
| `regression` | `round_clip` | $\text{clip}(\text{round}(\hat{y} \times 9), 0, 9)$ |

In [ ]:
DATASET_DIR = "/mnt/d/datasets/mnist"

# 3 task Predictor 출력 shape 확인
rng = np.random.default_rng(0)
x_dummy = rng.standard_normal((16, 784)).astype(np.float32)

for task in ["multiclass", "binary", "regression"]:
    model = MLP(task=task, seed=42)
    predictor = Predictor(model, get_task_spec(task))
    result = predictor.predict(x_dummy)
    print(f"[{task}] logits: {result['logits'].shape}, predictions: {result['predictions'].shape}",
          f"dtype={result['predictions'].dtype}")

In [ ]:
# multiclass: 예측값이 0~9 범위인지 확인
model_mc = MLP(task="multiclass", seed=42)
predictor_mc = Predictor(model_mc, get_task_spec("multiclass"))
result_mc = predictor_mc.predict(x_dummy)

preds = result_mc["predictions"]
assert preds.min() >= 0 and preds.max() <= 9
print("multiclass 예측값:", preds[:8])

In [ ]:
# binary: 예측값이 0 또는 1인지 확인
model_bi = MLP(task="binary", seed=42)
predictor_bi = Predictor(model_bi, get_task_spec("binary"))
result_bi = predictor_bi.predict(x_dummy)

preds_bi = result_bi["predictions"]
assert set(preds_bi.tolist()).issubset({0, 1})
print("binary 예측값:", preds_bi[:8])

In [ ]:
# regression: 예측값이 0~9 정수인지 확인
model_rg = MLP(task="regression", seed=42)
predictor_rg = Predictor(model_rg, get_task_spec("regression"))
result_rg = predictor_rg.predict(x_dummy)

preds_rg = result_rg["predictions"]
assert preds_rg.min() >= 0 and preds_rg.max() <= 9
print("regression 예측값:", preds_rg[:8])

## 2. Predictor와 Evaluator 차이

| 항목 | Evaluator | Predictor |
|---|---|---|
| 입력 | logit, target | logit |
| 처리 | loss/metric 계산 | task별 후처리 |
| 출력 | 숫자 지표 (loss, metric) | 해석 가능한 예측값 |
| 대상 | 모델 성능 측정 | 개별 샘플 예측 |

In [ ]:
# MNIST test set 첫 32개 샘플로 Predictor 실행
test_ds = MNISTDataset("test", "multiclass", dataset_dir=DATASET_DIR)
images = test_ds.images[:32]
labels_raw = test_ds.labels_raw[:32]

model_mc2 = MLP(task="multiclass", seed=42)
predictor_mc2 = Predictor(model_mc2, get_task_spec("multiclass"))
result2 = predictor_mc2.predict(images)

correct = (result2["predictions"] == labels_raw).sum()
print(f"32개 샘플 정확도 (초기화 파라미터): {correct}/32 = {correct/32:.3f}")
print("첫 8개 예측:", result2["predictions"][:8])
print("첫 8개 정답:", labels_raw[:8])

## 3. Visualizer — 예측 grid PNG 저장

`Visualizer.plot_predictions`는 N개 샘플의 이미지와 예측/정답 레이블을
격자 형태로 배치한 PNG 파일을 저장한다.

- 정답 → 초록색 타이틀
- 오답 → 빨간색 타이틀

$$\text{cols} = \min(8, N), \quad \text{rows} = \lceil N / \text{cols} \rceil$$

In [ ]:
# 임시 출력 디렉토리에 grid 저장
tmpdir = tempfile.mkdtemp()
visualizer = Visualizer(output_dir=tmpdir)

path = visualizer.plot_predictions(
    images=images,
    labels=labels_raw,
    predictions=result2["predictions"],
    filename="predictions_check.png",
    n=16,
)
assert os.path.exists(path)
print("grid PNG 저장 완료:", path)

In [ ]:
# 저장된 PNG 인라인 확인
from PIL import Image
img = Image.open(path)
plt.figure(figsize=(14, 4))
plt.imshow(img)
plt.axis("off")
plt.title("Predictor + Visualizer: 예측 grid (초기화 파라미터)")
plt.tight_layout()
plt.show()

## 4. Logger — 학습 로그 누적 및 CSV 내보내기

`Logger`는 `Trainer.fit`이 반환하는 로그를 컬럼별 리스트로 누적하고
dict 또는 CSV로 내보낸다.

내부 저장 구조:
- `epochs`: epoch 번호 리스트
- `losses`: epoch별 loss 리스트
- `metrics`: epoch별 metric 리스트

In [ ]:
# Logger append와 to_dict 기본 동작
logger = Logger()
logger.append(epoch=1, loss=1.234, metric=0.512)
logger.append(epoch=2, loss=0.891, metric=0.723)
logger.append(epoch=3, loss=0.671, metric=0.812)

d = logger.to_dict()
print("to_dict:", d)
assert d["epochs"] == [1, 2, 3]
assert len(d["losses"]) == 3
print("Logger append / to_dict 검증 통과")

In [ ]:
# Logger to_csv 확인
csv_path = os.path.join(tmpdir, "train_log.csv")
logger.to_csv(csv_path)
assert os.path.exists(csv_path)

with open(csv_path) as f:
    content = f.read()
print("CSV 내용:\n", content)
assert content.startswith("epoch,loss,metric")

## 5. Trainer + Logger 통합 — 5 epoch 학습 및 로그 저장

In [ ]:
task = "multiclass"
ts = get_task_spec(task)
train_ds = MNISTDataset("train", task, dataset_dir=DATASET_DIR)
test_ds2 = MNISTDataset("test", task, dataset_dir=DATASET_DIR)
train_loader = Dataloader(train_ds, batch_size=128, shuffle=True)
test_loader = Dataloader(test_ds2, batch_size=256, shuffle=False)

model_5 = MLP(task=task, seed=42)
opt_5 = SGD(model_5, lr=0.01)
trainer_5 = Trainer(model_5, opt_5, ts)
evaluator_5 = Evaluator(model_5, ts)
predictor_5 = Predictor(model_5, ts)
logger_5 = Logger()

full_logs = []
for epoch in range(1, 6):
    tr = trainer_5.fit(train_loader)
    te = evaluator_5.evaluate(test_loader)
    logger_5.append(epoch=epoch, loss=tr["loss"], metric=tr["metric"])
    full_logs.append({"epoch": epoch, "train": tr, "test": te})
    print(f"Epoch {epoch} | train loss={tr['loss']:.4f} metric={tr['metric']:.4f}"
          f" | test loss={te['loss']:.4f} metric={te['metric']:.4f}")

In [ ]:
# Logger 누적 확인
log_dict = logger_5.to_dict()
assert len(log_dict["epochs"]) == 5
print("Logger 누적 epochs:", log_dict["epochs"])

# CSV 저장
csv5_path = os.path.join(tmpdir, "train_log_5ep.csv")
logger_5.to_csv(csv5_path)
print("CSV 저장 완료:", csv5_path)

In [ ]:
# 학습 완료 후 Predictor로 예측 grid 생성
images_test = test_ds2.images[:32]
labels_raw_test = test_ds2.labels_raw[:32]

pred_result = predictor_5.predict(images_test)
vis5 = Visualizer(output_dir=tmpdir)
path5 = vis5.plot_predictions(
    images=images_test,
    labels=labels_raw_test,
    predictions=pred_result["predictions"],
    filename="predictions_5ep.png",
    n=16,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 학습 곡선
epochs_list = [log["epoch"] for log in full_logs]
axes[0].plot(epochs_list, [log["train"]["metric"] for log in full_logs], marker="o", label="train")
axes[0].plot(epochs_list, [log["test"]["metric"] for log in full_logs], marker="s", label="test")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("학습 곡선 (5 epochs)")
axes[0].legend()

# 예측 grid
axes[1].imshow(Image.open(path5))
axes[1].axis("off")
axes[1].set_title("예측 grid (5 epoch 학습 후)")

plt.tight_layout()
plt.show()

## 6. training_plots 연동

`Logger.to_dict()`의 반환값을 `save_training_plots`에 전달하면 학습 곡선 PNG를 저장할 수 있다.

In [ ]:
from src.utils.training_plots import plot_training_log

# training_plots은 full_logs (train + test 모두 포함) 형식을 받는다
plot_path = plot_training_log(full_logs, output_dir=tmpdir, filename="training_log_demo.png")
assert os.path.exists(plot_path)

plt.figure(figsize=(12, 4))
plt.imshow(Image.open(plot_path))
plt.axis("off")
plt.title("training_plots 출력")
plt.tight_layout()
plt.show()

print("training_plots 연동 검증 통과")

## 7. 요약

| 객체 | 역할 | 반환값 |
|---|---|---|
| `Predictor` | raw logit → task별 예측값 후처리 | `{"logits": ..., "predictions": ...}` |
| `Visualizer` | 예측/정답 grid PNG 저장 | 파일 경로 |
| `Logger` | epoch별 loss/metric 누적, CSV 내보내기 | `to_dict()`, `to_csv()` |

**주요 설계 원칙**
- `Predictor`는 `DECODE_FN` dict dispatch로 task별 lambda를 생성 시 바인딩한다
- `Visualizer`는 `T:{true}` / `P:{pred}` 타이틀을 정오답에 따라 초록/빨간색으로 구분한다
- `Logger`는 컬럼별 리스트로 분리하여 `losses`만 꺼내 그래프 그리기 등 부분 참조가 용이하다
- `Logger.to_dict()` 반환값은 `plot_training_log`의 입력이 아니라 직접 plot 용 dict이다
  (`plot_training_log`는 train/test가 분리된 `full_logs` 형식을 받는다)